## chargement

In [14]:
import pandas as pd

files = ["patient_administration",
          "clinical_encounters",
          "clinical_observations",
          "clinical_procedures",
          "clinical_notes",
          "cardiac_recordings",
          "cardiac_signal_samples"]

# Tout en texte : on veut voir les données brutes, sans conversion automatique
for name in files:
    tables = {name: pd.read_csv(f"health_not_clean/{name}.csv", dtype=str)}
    for name, df in tables.items():
        print(name, df.shape)


patient_administration (241, 10)
clinical_encounters (1069, 12)
clinical_observations (17736, 14)
clinical_procedures (311, 14)
clinical_notes (30, 11)
cardiac_recordings (44, 19)
cardiac_signal_samples (96925, 4)


## nettoyage

In [ ]:
def clean_text(df, exclude=[]):
    df = df.copy()
    for col in df.columns.difference(exclude):
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    return df

def lower_categories(df, cols):
    df = df.copy()
    for col in cols:
        before = df[col].nunique()
        df[col] = df[col].str.lower()
        if df[col].nunique() != before:
            print(f"  {col} : {before} -> {df[col].nunique()} valeurs distinctes")
    return df

def remove_duplicates(df, key):
    before = len(df)
    df = df.drop_duplicates()
    print(f"  doublons supprimés : {before - len(df)} | {key} unique : {df[key].is_unique}")
    return df

def to_dates(df, cols):
    df = df.copy()
    for col in cols:
        filled = df[col].notna().sum()
        iso = pd.to_datetime(df[col], format="ISO8601", errors="coerce")
        other = pd.to_datetime(df[col], format="mixed", dayfirst=True, errors="coerce")
        print(f"  {col} : {(iso.isna() & df[col].notna()).sum()} date(s) hors format ISO converties")
        df[col] = iso.fillna(other)
        assert df[col].notna().sum() == filled, f"dates perdues dans {col}"
    return df

def missing(df):
    m = df.isna().sum()
    m = m[m > 0]
    return pd.DataFrame({"manquants": m, "%": (m / len(df) * 100).round(1)})


## patients

In [3]:
pat = clean_text(tables["patient_administration"])
pat = lower_categories(pat, ["home_region", "registered_sex", "record_status", "occupation_text"])
pat = remove_duplicates(pat, "patient_number")
pat = to_dates(pat, ["date_of_birth", "registration_date"])
missing(pat)


  home_region : 6 -> 5 valeurs distinctes
  record_status : 4 -> 3 valeurs distinctes
  occupation_text : 44 -> 43 valeurs distinctes
  doublons supprimés : 1 | patient_number unique : True
  date_of_birth : 0 date(s) hors format ISO converties
  registration_date : 1 date(s) hors format ISO converties


,manquants,%


## consultations

In [4]:
enc = clean_text(tables["clinical_encounters"])
enc = lower_categories(enc, ["encounter_type", "referral_source", "service_unit", "disposition", "confidentiality_class"])
enc = remove_duplicates(enc, "encounter_number")
enc = to_dates(enc, ["encounter_start", "encounter_end", "follow_up_date"])
print("  fin avant début :", (enc["encounter_end"] < enc["encounter_start"]).sum())
missing(enc)


  encounter_type : 15 -> 10 valeurs distinctes
  referral_source : 8 -> 5 valeurs distinctes
  disposition : 7 -> 4 valeurs distinctes
  doublons supprimés : 2 | encounter_number unique : True
  encounter_start : 3 date(s) hors format ISO converties
  encounter_end : 3 date(s) hors format ISO converties
  follow_up_date : 1 date(s) hors format ISO converties
  fin avant début : 0


,manquants,%
referral_source,3,0.3
follow_up_date,240,22.5


## observations

In [5]:
obs = clean_text(tables["clinical_observations"])
obs = lower_categories(obs, ["observation_name", "interpretation_code", "quality_code", "text_value"])
obs = remove_duplicates(obs, "observation_number")
obs = to_dates(obs, ["observation_time"])

# Certaines valeurs sont écrites "84,9" ou "82.2 index" : on garde uniquement le nombre
raw = obs["numeric_value"]
not_numeric = raw.notna() & pd.to_numeric(raw, errors="coerce").isna()
print("  valeurs numériques mal écrites :", not_numeric.sum(), raw[not_numeric].head(5).tolist())
obs["numeric_value"] = pd.to_numeric(raw.str.replace(",", ".").str.extract(r"(-?\d+\.?\d*)")[0])
assert obs["numeric_value"].notna().sum() == raw.notna().sum()  # aucune valeur perdue
missing(obs)


  interpretation_code : 9 -> 4 valeurs distinctes
  quality_code : 3 -> 1 valeurs distinctes
  doublons supprimés : 2 | observation_number unique : True
  observation_time : 0 date(s) hors format ISO converties
  valeurs numériques mal écrites : 340 ['84,9', '82.2 index', '54,6', '87,8', '8.07 L/min']


,manquants,%
numeric_value,1870,10.5
text_value,15864,89.5
unit,1870,10.5
device_reference,4325,24.4


## actes et notes

In [6]:
proc = clean_text(tables["clinical_procedures"])
proc = lower_categories(proc, ["procedure_description", "authorization_class", "procedure_status", "outcome_code"])
proc = remove_duplicates(proc, "procedure_record_id")
proc = to_dates(proc, ["scheduled_time", "performed_time", "next_review_date"])
display(missing(proc))

# On ne touche pas au texte : ses retours à la ligne structurent le document
notes = clean_text(tables["clinical_notes"], exclude=["text"])
notes = remove_duplicates(notes, "document_id")
notes = to_dates(notes, ["document_date"])
missing(notes)


  authorization_class : 6 -> 4 valeurs distinctes
  outcome_code : 8 -> 6 valeurs distinctes
  doublons supprimés : 1 | procedure_record_id unique : True
  scheduled_time : 1 date(s) hors format ISO converties
  performed_time : 0 date(s) hors format ISO converties
  next_review_date : 0 date(s) hors format ISO converties


,manquants,%
material_reference,260,83.9
next_review_date,56,18.1


  doublons supprimés : 0 | document_id unique : True
  document_date : 0 date(s) hors format ISO converties


,manquants,%
superseded_document_id,30,100.0


##  enregistrements cardiaques

In [7]:
rec = clean_text(tables["cardiac_recordings"])
rec = remove_duplicates(rec, "recording_id")
rec = to_dates(rec, ["recording_date"])
print("  colonnes constantes :", rec.columns[rec.nunique() == 1].tolist())

# Le signal reste brut : on le convertit en nombres et on vérifie, sans le résumer
sig = tables["cardiac_signal_samples"].copy()
num_cols = ["time_ms", "channel_a_normalized", "channel_b_normalized"]
sig[num_cols] = sig[num_cols].apply(pd.to_numeric)
print("  manquants :", sig.isna().sum().sum(), "| doublons (recording_id, time_ms) :", sig.duplicated(["recording_id", "time_ms"]).sum())
n_samples = sig.groupby("recording_id").size()
print("  sample_count cohérent :", (rec.set_index("recording_id")["sample_count"].astype(int) == n_samples.reindex(rec["recording_id"]).values).all())


  doublons supprimés : 0 | recording_id unique : True
  recording_date : 0 date(s) hors format ISO converties
  colonnes constantes : ['channel_count', 'amplitude_unit', 'archive_status']
  manquants : 0 | doublons (recording_id, time_ms) : 0
  sample_count cohérent : True


## cohérence entre tables

In [8]:
for name, df in [("encounters", enc), ("observations", obs), ("procedures", proc), ("notes", notes), ("recordings", rec)]:
    unknown_pat = (~df["patient_number"].isin(pat["patient_number"])).sum()
    unknown_enc = (~df["encounter_number"].isin(enc["encounter_number"])).sum()
    print(f"{name} : {unknown_pat} patient(s) inconnu(s), {unknown_enc} consultation(s) inconnue(s)")


encounters : 0 patient(s) inconnu(s), 0 consultation(s) inconnue(s)
observations : 0 patient(s) inconnu(s), 0 consultation(s) inconnue(s)
procedures : 0 patient(s) inconnu(s), 0 consultation(s) inconnue(s)
notes : 0 patient(s) inconnu(s), 0 consultation(s) inconnue(s)
recordings : 0 patient(s) inconnu(s), 0 consultation(s) inconnue(s)


##  export des tables nettoyées

In [9]:
from pathlib import Path

OUT_DIR = Path("health_clean")
OUT_DIR.mkdir(exist_ok=True)  # crée data/health/health_clean/ s'il n'existe pas

clean_tables = {
    "patient_administration": pat,
    "clinical_encounters": enc,
    "clinical_observations": obs,
    "clinical_procedures": proc,
    "clinical_notes": notes,
    "cardiac_recordings": rec,
    "cardiac_signal_samples": sig,
}

for name, df in clean_tables.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name} : {df.shape} -> {OUT_DIR / (name + '.csv')}")


patient_administration : (240, 10) -> health_clean\patient_administration.csv
clinical_encounters : (1067, 12) -> health_clean\clinical_encounters.csv
clinical_observations : (17734, 14) -> health_clean\clinical_observations.csv
clinical_procedures : (310, 14) -> health_clean\clinical_procedures.csv
clinical_notes : (30, 11) -> health_clean\clinical_notes.csv
cardiac_recordings : (44, 19) -> health_clean\cardiac_recordings.csv
cardiac_signal_samples : (96925, 4) -> health_clean\cardiac_signal_samples.csv
